In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Instalação das bibliotecas

In [ ]:
%pip install -q transformers sentencepiece accelerate

Análise do dataset glaive_python para entrada no escopo do projeto

In [ ]:
import pandas as pd
import re

input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_iniciantes.parquet"
)

df = pd.read_parquet(input_path)

df_glaive = df[
    df["source"] == "glaive_python"
].copy()

print("=" * 80)
print("ANÁLISE DO GLAIVE PYTHON")
print("=" * 80)

print(f"Total: {len(df_glaive):,}")


patterns_outside_python = [
    r"```cpp",
    r"```c\+\+",
    r"\bc\+\+\b",
    r"\bcpp\b",
    r"\bc programming\b",
    r"\bc program\b",
    r"\bc language\b",
    r"#include\s*<[^>]+>",
    r"\bstd::\w+",
    r"\bcout\s*<<",
    r"\bcin\s*>>",
    r"\bnullptr\b",
    r"\bNode\s*\*",
    r"\bint\s+\w+\s*\([^)]*\)\s*\{",
    r"\bprintf\s*\(",
    r"\bscanf\s*\(",

    r"```java",
    r"```javascript",
    r"```js",
    r"```typescript",
    r"```ts",

    r"\bjava\b",
    r"\bjavascript\b",
    r"\btypescript\b",

    r"```ruby",
    r"```php",
    r"```swift",
    r"```kotlin",
    r"```rust",
    r"```go",

    r"\bruby\b",
    r"\bphp\b",
    r"\bswift\b",
    r"\bkotlin\b",
    r"\brust\b",
    r"\bgolang\b",
    r"\bgo language\b",

    r"```html",
    r"```css",

    r"\bhtml5?\b",
    r"\bcss3?\b",
    r"\bjquery\b",
    r"\bbootstrap\b",

    r"\breact\.?js\b",
    r"\bangular\.?js\b",
    r"\bvue\.?js\b",
    r"\bvuex\b",
    r"\bnode\.?js\b",
    r"\bjsx\b",
    r"\btsx\b",
    r"\buseState\s*\(",
    r"\buseEffect\s*\(",

    r"```sql",
    r"\bsql\b",
    r"\bmysql\b",
    r"\bpostgresql\b",
    r"\bpostgres\b",
    r"\bmongodb\b",
    r"\boracle database\b",
    r"\bsql server\b",

    r"\bcuda\b",
    r"\bopencl\b",
    r"\bc\+\+\s*amp\b",

    r"\bmachine learning\b",
    r"\bdeep learning\b",
    r"\bneural network(?:s)?\b",
    r"\bgradient descent\b",
    r"\bgradient boosting\b",
    r"\brandom forest\b",
    r"\bclassification model\b",
    r"\bregression model\b",
    r"\bclustering\b",
    r"\bscikit[- ]learn\b",

    r"\bnatural language processing\b",
    r"\bnlp\b",
    r"\btext summarization\b",
    r"\bsentiment analysis\b",

    r"\bbig data\b",
    r"\bdata mining\b",
    r"\bhadoop\b",
    r"\bspark\b",
    r"\bmllib\b",
    r"\bpyspark\b",
    r"\bvowpal wabbit\b",

    r"\brabbitmq\b",
    r"\bpika\b",
    r"\bkubernetes\b",
    r"\bdocker\b",
    r"\bmicroservices?\b",
    r"\bdistributed computing\b",
    r"\bdistributed system\b",
    r"\bnetwork programming\b",

    r"\brest api\b",
    r"\brestful api\b",
    r"\brest service\b",
    r"\bapi endpoint\b",
    r"```xml",
    r"\bxml\b",
]

outside_python_pattern = re.compile(
    "|".join(patterns_outside_python),
    flags=re.IGNORECASE
)

questions = (
    df_glaive["question"]
    .fillna("")
    .astype(str)
)

answers = (
    df_glaive["answer"]
    .fillna("")
    .astype(str)
)

combined_text = questions + "\n" + answers

df_glaive["outside_python"] = combined_text.str.contains(
    outside_python_pattern,
    na=False
)

retained = df_glaive[
    ~df_glaive["outside_python"]
].copy()

removed = df_glaive[
    df_glaive["outside_python"]
].copy()

print("\n")
print("=" * 80)
print("RESUMO")
print("=" * 80)

print(f"Total analisado:       {len(df_glaive):,}")
print(f"Manter:                {len(retained):,}")
print(f"Remover:               {len(removed):,}")

retained_percentage = (
    len(retained) / len(df_glaive) * 100
    if len(df_glaive) > 0
    else 0
)

removed_percentage = (
    len(removed) / len(df_glaive) * 100
    if len(df_glaive) > 0
    else 0
)

print(f"\nPercentual mantido:    {retained_percentage:.2f}%")
print(f"Percentual removido:   {removed_percentage:.2f}%")


def show_sample(df_sample, title, quantity=30):
    print("\n")
    print("=" * 80)
    print(title)
    print("=" * 80)

    if len(df_sample) == 0:
        print("Nenhum registro encontrado.")
        return

    sample = df_sample.sample(
        n=min(quantity, len(df_sample)),
        random_state=42
    )

    for index, (_, row) in enumerate(
        sample.iterrows(),
        start=1
    ):
        print("\n" + "-" * 80)
        print(f"REGISTRO {index}")

        print("\nPERGUNTA:")
        print(row["question"])

        print("\nRESPOSTA:")
        print(str(row["answer"])[:1200])


show_sample(
    retained,
    "AMOSTRA — MANTIDOS",
    quantity=30
)

show_sample(
    removed,
    "AMOSTRA — REMOVIDOS",
    quantity=30
)

df_without_glaive = df[
    df["source"] != "glaive_python"
].copy()

df_final = pd.concat(
    [df_without_glaive, retained],
    ignore_index=True
)

df_final = df_final.drop(
    columns=["outside_python"],
    errors="ignore"
)

output_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_iniciantes_filtrado.parquet"
)

df_final.to_parquet(
    output_path,
    index=False
)

print("\n")
print("=" * 80)
print("ARQUIVO FINAL SALVO")
print("=" * 80)

print(f"Arquivo: {output_path}")
print(f"Total final: {len(df_final):,}")

print("\nQuantidade por fonte:")
print(df_final["source"].value_counts())

print("\nGlaive Python após limpeza:")
print(
    (df_final["source"] == "glaive_python").sum()
)

Análise final do dataset glaive_python para entrada no escopo do projeto

In [ ]:
import pandas as pd
import re

input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_iniciantes_filtrado.parquet"
)

df = pd.read_parquet(input_path)

df_glaive = df[
    df["source"] == "glaive_python"
].copy()

print("=" * 80)
print("ANÁLISE DO GLAIVE PYTHON")
print("=" * 80)

print(f"Total: {len(df_glaive):,}")

advanced_patterns = [
    r"\bo\s*\(\s*log\s*n\s*\)",
    r"\bo\s*\(\s*n\s*log\s*n\s*\)",
    r"\btime complexity\b",
    r"\bspace complexity\b",
    r"\bcomputational complexity\b",

    r"\bdynamic programming\b",
    r"\bmatrix exponentiation\b",
    r"\bdivide and conquer\b",
    r"\bbacktracking\b",
    r"\bgreedy algorithm\b",
    r"\btraveling salesman\b",
    r"\btravelling salesman\b",

    r"\bquicksort\b",
    r"\bmergesort\b",
    r"\bbinary tree\b",
    r"\bavl tree\b",
    r"\bred[- ]black tree\b",
    r"\bgraph algorithm\b",
    r"\bdepth[- ]first search\b",
    r"\bbreadth[- ]first search\b",
    r"\bshortest path\b",
    r"\bdijkstra(?:'s)? algorithm\b",
    r"\bbellman[- ]ford\b",
    r"\bfloyd[- ]warshall\b",
    r"\bmaximum subarray\b",
    r"\bkadane(?:'s)? algorithm\b",
    r"\bedit distance\b",
    r"\blevenshtein\b",
    r"\bspiral (?:matrix|order)\b",

    r"\bbitwise\b",
    r"\bbit manipulation\b",
    r"\bbit shift\b",
    r"\bbitwise shift\b",

    r"\bmetaclass(?:es)?\b",
    r"\bmetaprogram(?:ming)?\b",
    r"\bdescriptor(?:s)?\b",
    r"\bbytecode\b",
    r"\b__slots__\b",
    r"\b__getattribute__\b",
    r"\b__setattr__\b",

    r"\bmultiprocessing\b",
    r"\bmultithread(?:ed|ing)?\b",
    r"\bconcurrency\b",
    r"\bcoroutine(?:s)?\b",
]

out_of_scope_patterns = [
    r"\bc\+\+\b",
    r"\bcpp\b",
    r"\bc programming\b",
    r"\bc program\b",
    r"\bc language\b",

    r"\bjava\b",
    r"\bjavascript\b",
    r"\btypescript\b",
    r"\bruby\b",
    r"\bphp\b",
    r"(?<!\w)c#(?!\w)",
    r"\bcsharp\b",
    r"\bgolang\b",
    r"\bgo language\b",
    r"\bswift\b",
    r"\bkotlin\b",
    r"\brust\b",

    r"\bhtml\b",
    r"\bhtml5\b",
    r"\bcss\b",
    r"\bcss3\b",
    r"\bjquery\b",
    r"\bbootstrap\b",

    r"\bmongodb\b",
    r"\bmysql\b",
    r"\bpostgresql\b",
    r"\bpostgres\b",
    r"\boracle database\b",
    r"\bsql server\b",

    r"```glsl",
    r"\bglsl\b",
    r"\bopengl\b",
    r"\bshader\b",
    r"\bshaders\b",
    r"\bfragment shader\b",
    r"\bvertex shader\b",
    r"\bgeometry shader\b",
    r"\bcompute shader\b",

    r"\bcuda\b",
    r"\bopencl\b",
    r"\bc\+\+ amp\b",

    r"\bmachine learning\b",
    r"\bdeep learning\b",
    r"\bneural network(?:s)?\b",
    r"\bgradient descent\b",
    r"\bgradient boosting\b",
    r"\brandom forest\b",
    r"\bclassification model\b",
    r"\bregression model\b",
    r"\bclustering\b",
    r"\bscikit[- ]learn\b",

    r"\bnatural language processing\b",
    r"\bnlp\b",
    r"\btext summarization\b",
    r"\bsentiment analysis\b",

    r"\bbig data\b",
    r"\bdata mining\b",
    r"\bhadoop\b",
    r"\bspark\b",
    r"\bmllib\b",
    r"\bpyspark\b",
    r"\bvowpal wabbit\b",

    r"\brabbitmq\b",
    r"\bpika\b",
    r"\bkubernetes\b",
    r"\bdocker\b",
    r"\bmicroservices?\b",
    r"\bdistributed computing\b",
    r"\bdistributed system\b",
    r"\bnetwork programming\b",

    r"\brest api\b",
    r"\brestful api\b",
    r"\brest service\b",
    r"\bapi endpoint\b",

    r"\bxml\b",
]

advanced_pattern = re.compile(
    "|".join(advanced_patterns),
    flags=re.IGNORECASE
)

out_of_scope_pattern = re.compile(
    "|".join(out_of_scope_patterns),
    flags=re.IGNORECASE
)

questions = (
    df_glaive["question"]
    .fillna("")
    .astype(str)
    .str.lower()
)

answers = (
    df_glaive["answer"]
    .fillna("")
    .astype(str)
    .str.lower()
)

combined_text = questions + " " + answers

df_glaive["advanced_content"] = combined_text.str.contains(
    advanced_pattern,
    na=False
)

df_glaive["out_of_scope"] = combined_text.str.contains(
    out_of_scope_pattern,
    na=False
)

df_glaive["classification"] = "adequado"

df_glaive.loc[
    df_glaive["out_of_scope"],
    "classification"
] = "fora_escopo"

df_glaive.loc[
    df_glaive["advanced_content"] &
    ~df_glaive["out_of_scope"],
    "classification"
] = "avancado"

appropriate_records = df_glaive[
    df_glaive["classification"] == "adequado"
]

advanced_records = df_glaive[
    df_glaive["classification"] == "avancado"
]

out_of_scope_records = df_glaive[
    df_glaive["classification"] == "fora_escopo"
]

print("\n")
print("=" * 80)
print("RESUMO")
print("=" * 80)

print(f"Adequados:       {len(appropriate_records):,}")
print(f"Avançados:       {len(advanced_records):,}")
print(f"Fora do escopo:  {len(out_of_scope_records):,}")


def show_sample(df_sample, title, quantity=30):
    print("\n")
    print("=" * 80)
    print(title)
    print("=" * 80)

    if len(df_sample) == 0:
        print("Nenhum registro encontrado.")
        return

    sample = df_sample.sample(
        n=min(quantity, len(df_sample)),
        random_state=42
    )

    for index, (_, row) in enumerate(
        sample.iterrows(),
        start=1
    ):
        print("\n" + "-" * 80)
        print(f"REGISTRO {index}")

        print("\nPERGUNTA:")
        print(row["question"])

        print("\nRESPOSTA:")
        print(str(row["answer"])[:1200])


show_sample(
    appropriate_records,
    "AMOSTRA — ADEQUADOS",
    quantity=30
)

show_sample(
    advanced_records,
    "AMOSTRA — AVANÇADOS",
    quantity=30
)

show_sample(
    out_of_scope_records,
    "AMOSTRA — FORA DO ESCOPO",
    quantity=30
)

df_glaive_final = df_glaive[
    df_glaive["classification"] != "fora_escopo"
].copy()

df_glaive_final = df_glaive_final.drop(
    columns=[
        "advanced_content",
        "out_of_scope",
        "classification"
    ],
    errors="ignore"
)

output_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_iniciantes_filtrado.parquet"
)

df_glaive_final.to_parquet(
    output_path,
    index=False
)

print("\n")
print("=" * 80)
print("DATASET SALVO")
print("=" * 80)

print(f"Registros salvos: {len(df_glaive_final):,}")
print(f"Registros removidos: {len(out_of_scope_records):,}")
print(f"Arquivo: {output_path}")

Análise do arquivo python_docs.parquet

In [ ]:
import pandas as pd
import re

input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs.parquet"
)

df = pd.read_parquet(input_path)

print("=" * 80)
print("ANÁLISE TEMÁTICA — PYTHON DOCS")
print("=" * 80)

print(f"Total de registros: {len(df):,}")
print(f"Colunas: {list(df.columns)}")

print("\n" + "=" * 80)
print("VALORES AUSENTES")
print("=" * 80)

print(df[["question", "answer"]].isna().sum())

questions = (
    df["question"]
    .fillna("")
    .astype(str)
    .str.lower()
)

answers = (
    df["answer"]
    .fillna("")
    .astype(str)
    .str.lower()
)

combined_text = questions + " " + answers

topics = {
    "Variáveis e atribuição": [
        r"\bvariável\b",
        r"\bvariáveis\b",
        r"\batribuição\b",
        r"\batribuir\b",
    ],

    "Tipos de dados": [
        r"\btipos? de dados?\b",
        r"\bint\b",
        r"\binteger\b",
        r"\bfloat\b",
        r"\bbooleano\b",
        r"\bbool\b",
        r"\bnenhum\b",
        r"\bnone\b",
    ],

    "Strings": [
        r"\bstring\b",
        r"\bstrings\b",
        r"\btexto\b",
        r"\btextos\b",
        r"\bstr\b",
    ],

    "Listas": [
        r"\blista\b",
        r"\blistas\b",
        r"\blist\b",
    ],

    "Tuplas": [
        r"\btupla\b",
        r"\btuplas\b",
        r"\btuple\b",
    ],

    "Dicionários": [
        r"\bdicionário\b",
        r"\bdicionários\b",
        r"\bdict\b",
    ],

    "Conjuntos": [
        r"\bconjunto\b",
        r"\bconjuntos\b",
        r"\bset\b",
    ],

    "Condicionais": [
        r"\bcondicional\b",
        r"\bcondicionais\b",
        r"\bif\b",
        r"\belif\b",
        r"\belse\b",
    ],

    "Loops e controle de repetição": [
        r"\bloop\b",
        r"\bloops\b",
        r"\blaço\b",
        r"\blaços\b",
        r"\brepetição\b",
        r"\bfor\b",
        r"\bwhile\b",
        r"\bbreak\b",
        r"\bcontinue\b",
        r"\bpass\b",
    ],

    "Funções": [
        r"\bfunção\b",
        r"\bfunções\b",
        r"\bparâmetro\b",
        r"\bparâmetros\b",
        r"\bargumento\b",
        r"\bargumentos\b",
        r"\breturn\b",
        r"\bretorno\b",
    ],

    "Exceções e erros": [
        r"\bexceção\b",
        r"\bexceções\b",
        r"\berro\b",
        r"\berros\b",
        r"\btry\b",
        r"\bexcept\b",
        r"\bfinally\b",
        r"\braise\b",
    ],

    "Módulos e importação": [
        r"\bmódulo\b",
        r"\bmódulos\b",
        r"\bimportação\b",
        r"\bimportar\b",
        r"\bimport\b",
        r"\bfrom\b",
    ],

    "Arquivos": [
        r"\barquivo\b",
        r"\barquivos\b",
        r"\bopen\b",
        r"\bleitura\b",
        r"\bescrita\b",
        r"\bleitura de arquivo\b",
    ],

    "Classes e objetos": [
        r"\bclasse\b",
        r"\bclasses\b",
        r"\bobjeto\b",
        r"\bobjetos\b",
        r"\binstância\b",
        r"\binstâncias\b",
    ],

    "Programação Orientada a Objetos": [
        r"\bprogramação orientada a objetos\b",
        r"\bpoo\b",
        r"\bherança\b",
        r"\bpolimorfismo\b",
        r"\bencapsulamento\b",
    ],

    "Compreensão de listas": [
        r"\bcompreensão de lista\b",
        r"\bcompreensão de listas\b",
        r"\blist comprehension\b",
    ],

    "Expressões lambda": [
        r"\blambda\b",
        r"\bfunção anônima\b",
        r"\bfunções anônimas\b",
    ],

    "Geradores": [
        r"\bgerador\b",
        r"\bgeradores\b",
        r"\bgenerator\b",
        r"\byield\b",
    ],

    "Decoradores": [
        r"\bdecorador\b",
        r"\bdecoradores\b",
        r"\bdecorator\b",
        r"\bdecorators\b",
    ],

    "Bibliotecas": [
        r"\bbiblioteca\b",
        r"\bbibliotecas\b",
        r"\bstdlib\b",
        r"\bbiblioteca padrão\b",
    ],

    "Entrada e saída": [
        r"\bentrada\b",
        r"\bsaída\b",
        r"\binput\b",
        r"\bprint\b",
        r"\bimpressão\b",
    ],

    "Operadores": [
        r"\boperador\b",
        r"\boperadores\b",
        r"\baritmético\b",
        r"\baritméticos\b",
        r"\bcomparação\b",
        r"\blógico\b",
        r"\blógicos\b",
        r"\brelacional\b",
        r"\brelacionais\b",
        r"\batribuição\b",
        r"\bassociação\b",
    ],

    "Estruturas de dados": [
        r"\bpilha\b",
        r"\bpilhas\b",
        r"\bstack\b",
        r"\bstacks\b",
        r"\bfila\b",
        r"\bfilas\b",
        r"\bqueue\b",
        r"\bqueues\b",
    ],

    "Match e case": [
        r"\bmatch\b",
        r"\bcase\b",
        r"\bpattern matching\b",
        r"\bcorrespondência de padrões\b",
    ],

    "Pacotes e pip": [
        r"\bpip\b",
        r"\bpip3\b",
        r"\binstalar pacote\b",
        r"\binstalação de pacote\b",
        r"\binstalação de biblioteca\b",
        r"\bgerenciamento de pacotes\b",
        r"\bpackage manager\b",
    ],

    "Ambientes virtuais": [
        r"\bambiente virtual\b",
        r"\bambientes virtuais\b",
        r"\bvirtual environment\b",
        r"\bvenv\b",
        r"\bvirtualenv\b",
    ],

    "Controle de loops": [
        r"\bbreak\b",
        r"\bcontinue\b",
        r"\bpass\b",
    ],
}

results = []

for topic, patterns in topics.items():
    topic_pattern = re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE
    )

    mask = combined_text.str.contains(
        topic_pattern,
        na=False
    )

    count = mask.sum()

    results.append({
        "tema": topic,
        "registros": count,
        "percentual": round(
            (count / len(df)) * 100,
            2
        ) if len(df) > 0 else 0
    })

coverage = pd.DataFrame(results)

coverage = coverage.sort_values(
    by="registros",
    ascending=False
)

print("\n" + "=" * 80)
print("COBERTURA TEMÁTICA")
print("=" * 80)

print(
    coverage.to_string(index=False)
)

mask_any_topic = pd.Series(
    False,
    index=df.index
)

for patterns in topics.values():
    topic_pattern = re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE
    )

    mask_any_topic |= combined_text.str.contains(
        topic_pattern,
        na=False
    )

records_without_topic = df[
    ~mask_any_topic
]

print("\n" + "=" * 80)
print("REGISTROS SEM TEMA IDENTIFICADO")
print("=" * 80)

print(
    f"Total: {len(records_without_topic):,}"
)

print(
    f"Percentual: "
    f"{(len(records_without_topic) / len(df)) * 100:.2f}%"
    if len(df) > 0
    else "Percentual: 0.00%"
)

if len(records_without_topic) > 0:
    print("\n" + "=" * 80)
    print("AMOSTRA — SEM TEMA IDENTIFICADO")
    print("=" * 80)

    sample = records_without_topic.sample(
        n=min(20, len(records_without_topic)),
        random_state=42
    )

    for index, (_, row) in enumerate(sample.iterrows(), start=1):
        print("\n" + "-" * 80)
        print(f"REGISTRO {index}")

        print("\nPERGUNTA:")
        print(row["question"])

        print("\nRESPOSTA:")
        print(str(row["answer"])[:1000])

        print("\nCÓDIGO")
        print(str(row["code"])[:1000])

Limpeza de duplicatas

In [ ]:
from pathlib import Path
import pandas as pd

INPUT_PATH = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs.parquet"
)

OUTPUT_PATH = Path(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/python_docs_sem_duplicatas.parquet"
)

df = pd.read_parquet(INPUT_PATH)

print("=" * 80)
print("REMOÇÃO DE PERGUNTAS DUPLICADAS")
print("=" * 80)

print(f"Registros originais: {len(df):,}")

df["_normalized_question"] = (
    df["question"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

duplicates = df[
    df.duplicated(
        subset=["_normalized_question"],
        keep=False
    )
].sort_values(
    "_normalized_question"
)

print(
    f"Registros envolvidos em perguntas duplicadas: "
    f"{len(duplicates):,}"
)

print(
    f"Perguntas duplicadas distintas: "
    f"{duplicates['_normalized_question'].nunique():,}"
)

df_clean = df.drop_duplicates(
    subset=["_normalized_question"],
    keep="first"
).reset_index(
    drop=True
)

df_clean = df_clean.drop(
    columns=["_normalized_question"]
)

removed_count = len(df) - len(df_clean)

print()
print(f"Registros após limpeza: {len(df_clean):,}")
print(f"Perguntas removidas: {removed_count:,}")

df_clean.to_parquet(
    OUTPUT_PATH,
    index=False
)

print()
print("Arquivo limpo salvo em:")
print(OUTPUT_PATH)

df_validation = df_clean.copy()

df_validation["_normalized_question"] = (
    df_validation["question"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

remaining_duplicates = df_validation.duplicated(
    subset=["_normalized_question"]
).sum()

print()
print("=" * 80)
print("VERIFICAÇÃO FINAL")
print("=" * 80)

print(
    f"Duplicatas ignorando maiúsculas/minúsculas: "
    f"{remaining_duplicates:,}"
)

if remaining_duplicates == 0:
    print("Nenhuma pergunta duplicada restante.")
else:
    print("Ainda existem perguntas duplicadas.")

Dataset limpo sem duplicatas

In [ ]:
import pandas as pd
import re

input_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_ensino_limpo.parquet"
)

df = pd.read_parquet(input_path)

print(f"Original: {len(df)} registros")

combined_text = (
    df["question"].fillna("") + " " +
    df["answer"].fillna("") + " " +
    df["code"].fillna("")
).str.lower()

exclude_patterns = [
    r"\bmachine\s+learning\b",
    r"\bdeep\s+learning\b",
    r"\bneural\s+network",
    r"\btensorflow\b",
    r"\bpytorch\b",
    r"\bkeras\b",
    r"\bscikit[- ]?learn\b",
    r"\bsklearn\b",
    r"\bartificial\s+intelligence\b",
    r"\bnlp\b",
    r"\bnatural\s+language\s+processing\b",
    r"\bcomputer\s+vision\b",
    r"\bopencv\b",
    r"\bxgboost\b",
    r"\brandom\s+forest\b",
    r"\bdecision\s+tree\b",
    r"\bsupport\s+vector\b",
    r"\bk[- ]?means\b",
    r"\bnearest\s*neighbors?\b",
    r"\bhyperparameter\b",
    r"\bcross[- ]?validation\b",
    r"\bf1[- ]?score\b",
    r"\bconfusion\s+matrix\b",
    r"\bfeature\s+engineering\b",
    r"\bpandas\b",
    r"\bdataframe\b",
    r"\bnumpy\b",
    r"\bmatplotlib\b",
    r"\bseaborn\b",
    r"\bplotly\b",
    r"\bstatsmodels\b",
    r"\bdata\s+science\b",
    r"\bdata\s+analysis\b",
    r"\bdata\s+visualization\b",
    r"\bscipy\b",
    r"\bpillow\b",
    r"\bpil\b",
    r"\bimage\s+processing\b",

    r"\bdjango\b",
    r"\bflask\b",
    r"\bfastapi\b",
    r"\bbeautifulsoup\b",
    r"\bscrapy\b",
    r"\bselenium\b",
    r"\brequests\b",
    r"\burllib\b",
    r"\bhttp\.client\b",
    r"\brest\s+api\b",
    r"\bweb\s+scraping\b",
    r"\bwebsocket\b",
    r"\bsocket\b",
    r"\btcp\b",
    r"\budp\b",

    r"\basyncio\b",
    r"\bmultiprocessing\b",
    r"\bthreading\b",
    r"\bconcurrent\.futures\b",
    r"\bsubprocess\b",
    r"\bos\.system\b",
    r"\boperating\s+system\b",

    r"\bbinary\s+tree\b",
    r"\bbinary\s+search\b",
    r"\bavl\s+tree\b",
    r"\bred[- ]?black\b",
    r"\bb[- ]?tree\b",
    r"\btrie\b",
    r"\bgraph\s+theory\b",
    r"\bdirected\s+graph\b",
    r"\bundirected\s+graph\b",
    r"\bbreadth[- ]?first\s+search\b",
    r"\bdepth[- ]?first\s+search\b",
    r"\bbfs\b",
    r"\bdfs\b",

    r"\bbig[- ]?o\b",
    r"\bbig\s+o\s+notation\b",
    r"\bo\(1\)",
    r"\bo\(n\)",
    r"\bo\(n\^2\)",
    r"\bo\(n\*\*2\)",
    r"\bo\(log",
    r"\bo\(n\s*log",
    r"\btime\s+complexity\b",
    r"\bspace\s+complexity\b",
    r"\basymptotic\b",
    r"\bcomputational\s+complexity\b",

    r"\bsqlalchemy\b",
    r"\bsqlite\b",
    r"\bmysql\b",
    r"\bpostgresql\b",
    r"\bmongodb\b",
    r"\bredis\b",
    r"\bpymongo\b",
    r"\borm\b",
    r"\bdatabase\b",

    r"\btkinter\b",
    r"\bpyqt\b",
    r"\bkivy\b",
    r"\bpygame\b",
    r"\bgui\b",

    r"\bblockchain\b",
    r"\bbitcoin\b",
    r"\bethereum\b",
    r"\bdocker\b",
    r"\bkubernetes\b",
    r"\baws\b",
    r"\bazure\b",
    r"\bgcp\b",
    r"\bcrypto\b",
    r"\bencryption\b",
]

exclude_pattern = "|".join(exclude_patterns)

exclude_mask = combined_text.str.contains(
    exclude_pattern,
    regex=True,
    na=False
)

df_clean = df[
    ~exclude_mask
].copy().reset_index(drop=True)

print(f"Removidos: {exclude_mask.sum()}")
print(f"Mantidos:  {len(df_clean)}")

df_clean = df_clean[
    (df_clean["question"].str.len() >= 20) &
    (df_clean["answer"].str.len() >= 30)
].reset_index(drop=True)

print(f"Após filtro de tamanho mínimo: {len(df_clean)}")

output_path = (
    "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_ensino_limpo.parquet"
)

df_clean.to_parquet(
    output_path,
    index=False
)

print("\nArquivo salvo no Google Drive:")
print(f"  - {output_path}")

print("\n--- Amostra de perguntas mantidas ---")

for question in df_clean["question"].sample(min(8, len(df_clean)), random_state=42):
    print(f"• {question[:140]}...")

Visualização do dataset sem duplicatas

In [ ]:
import pandas as pd

path = "/content/drive/MyDrive/TCC_Chatbot/datasets/curated/banco_python_ensino_limpo.parquet"

df = pd.read_parquet(path)

print("========== RESUMO ==========")
print(f"Total de registros: {len(df)}")
print(f"Total de colunas: {len(df.columns)}")

print("\n========== COLUNAS ==========")
print(list(df.columns))

print("\n========== SOURCES ==========")
print(df["source"].value_counts())

print("\n========== NULOS ==========")
print(df.isna().sum())

print("\n========== AMOSTRA ==========")
print(df[["source", "question", "answer"]].sample(10, random_state=42).to_string(index=False))